# Preparation

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/vedauv/what-makes-a-yt-video-go-viral-12000-vids/README.md
/kaggle/input/datasets/vedauv/what-makes-a-yt-video-go-viral-12000-vids/datasheet.md
/kaggle/input/datasets/vedauv/what-makes-a-yt-video-go-viral-12000-vids/viral_videos.csv


In [2]:
import os
import sklearn
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, silhouette_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

In [3]:
import os
main_csv_local_path = 'viral_videos.csv'
data_dir = ''

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        if filename == main_csv_local_path:
            data_dir = os.path.join(dirname, filename)
            print(data_dir)

/kaggle/input/datasets/vedauv/what-makes-a-yt-video-go-viral-12000-vids/viral_videos.csv


# Exploratory Data Analysis (EDA)

In [4]:
# Load the CSV from the directory into DataFrame
try:
    df = pd.read_csv(data_dir, encoding='utf-8')
except UnicodeDecodeError:
    df = pd.read_csv(data_dir, encoding='latin1')

# Analyze shape, missing values, duplicates, numeric, and categorical columns
print("Dataset shape: ", df.shape)
print("Missing values summary: ", df.isnull().sum().to_dict())
print("Duplicate rows: ", df.duplicated().sum())

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()
print("Numeric columns count: ", len(numeric_cols))
print("Categorical columns count: ", len(categorical_cols))

# Remove a copy named df_clean without changing its directory and/or original DataFrame
df_clean = df.copy()
df_clean = df_clean.drop_duplicates()

for col in numeric_cols:
    if df_clean[col].isnull().sum() > 0:
        df_clean[col].fillna(df_clean[col].median(), inplace=True)

for col in categorical_cols:
    if df_clean[col].isnull().sum() > 0:
        fill_val = df_clean[col].mode()[0] if not df_clean[col].mode().empty else 'Missing'
        df_clean[col].fillna(fill_val, inplace=True)

# Compact exploratory data analysis (EDA)'s summaries and guarded plots
print(df_clean.describe())
plt.figure(figsize=(6,4))
sns.histplot(df_clean['views'], bins=30, kde=True)
plt.title("Distribution of YouTube Views")
plt.tight_layout()
plt.savefig('youtube_views_distribution.png')
plt.close()

# Assign target_col function
target_col = 'viral' if 'viral' in df_clean.columns else None
print("Target column: ", target_col)

# Feature engineering
for col in ['video_id', 'category', 'upload_day']:
    if col in df_clean.columns:
        df_clean[f"{col}__len"] = df_clean[col].astype(str).apply(len)
        df_clean[f"{col}__words"] = df_clean[col].astype(str).apply(lambda x: len(x.split()))

# Problem type detection & handle functions
def detect_problem_type(data, target):
    if target is None or target not in data.columns:
        return 'unsupervised'
    series = data[target]
    if pd.api.types.is_bool_dtype(series) or pd.api.types.is_object_dtype(series) or pd.api.types.is_categorical_dtype(series):
        return 'classification'
    if pd.api.types.is_numeric_dtype(series):
        if series.nunique() <= 15 or (series.nunique() / len(series) < 0.01):
            return 'classification'
        else:
            return 'regression'
    return 'classification'

problem_type = detect_problem_type(df_clean, target_col)
print("Detected Problem Type:", problem_type)

def build_preprocessor(num_cols, cat_cols):
    num_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])
    cat_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])
    return ColumnTransformer(transformers=[
        ('num', num_transformer, num_cols),
        ('cat', cat_transformer, cat_cols)
    ])

def evaluate_classification(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    return {'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1}

def evaluate_regression(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    return {'mae': mae, 'mse': mse, 'rmse': rmse, 'r2': r2}

def run_kmeans(X_data, max_k=8):
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_data)
    inertias = []
    sil_scores = []
    k_range = range(2, min(max_k + 1, len(X_scaled)))
    for k in k_range:
        km = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = km.fit_predict(X_scaled)
        inertias.append(km.inertia_)
        sil_scores.append(silhouette_score(X_scaled, labels))
    best_k = k_range[np.argmax(sil_scores)] if sil_scores else 2
    best_km = KMeans(n_clusters=best_k, random_state=42, n_init=10)
    best_km.fit(X_scaled)
    return best_km, inertias, sil_scores

model_results = []
best_model = None

# Modeling pipeline & evaluation
if problem_type in ['classification', 'regression']:
    feature_cols = [c for c in df_clean.columns if c != target_col and c != 'video_id']
    X = df_clean[feature_cols]
    y = df_clean[target_col]

    num_features = X.select_dtypes(include=[np.number]).columns.tolist()
    cat_features = X.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()

    preprocessor = build_preprocessor(num_features, cat_features)
    stratify_val = y if problem_type == 'classification' and y.nunique() > 1 else None
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=stratify_val)

if problem_type == 'classification':
        models = {
            'LogisticRegression': LogisticRegression(max_iter=1000, random_state=42),
            'KNeighborsClassifier': KNeighborsClassifier(),
            'DecisionTreeClassifier': DecisionTreeClassifier(random_state=42),
            'RandomForestClassifier': RandomForestClassifier(random_state=42, n_estimators=50)
        }
else:
        models = {
            'LinearRegression': LinearRegression(),
            'KNeighborsRegressor': KNeighborsRegressor(),
            'DecisionTreeRegressor': DecisionTreeRegressor(random_state=42),
            'RandomForestRegressor': RandomForestRegressor(random_state=42, n_estimators=50)
        }
        best_score = -float('inf')
    
for name, model in models.items():
        try:
            pipe = Pipeline(steps=[('preprocessor', preprocessor), ('model', model)])
            pipe.fit(X_train, y_train)
            y_pred = pipe.predict(X_test)

            if problem_type == 'classification':
                metrics = evaluate_classification(y_test, y_pred)
                metrics['model'] = name
                model_results.append(metrics)
                if metrics['f1'] > best_score:
                    best_score = metrics['f1']
                    best_model = pipe
                else:
                    metrics = evaluate_regression(y_test, y_pred)
                    metrics['model'] = name
                    model_results.append(metrics)
                    if metrics['r2'] > best_score:
                       best_score = metrics['r2']
                       best_model = pipe
        except Exception as e:
            print(f"Model {name} failed: {e}")
        results_df = pd.DataFrame(model_results)
    
        if not results_df.empty:
            sort_col = 'f1' if problem_type == 'classification' else 'r2'
            results_df = results_df.sort_values(by=sort_col, ascending=False).reset_index(drop=True)
        print("Model Evaluation Results:\n", results_df)

# Modeling visualization
if problem_type == 'classification' and best_model is not None:
    y_pred_best = best_model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred_best)
    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title("Confusion Matrix - Best Model")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.tight_layout()
    plt.savefig('confusion_matrix.png')
    plt.close()
elif problem_type == 'regression' and best_model is not None:
    y_pred_best = best_model.predict(X_test)
    plt.figure(figsize=(5,4))
    plt.scatter(y_test, y_pred_best, alpha=0.3)
    plt.xlabel("Actual")
    plt.ylabel("Predicted")
    plt.title("Actual Vs. Predicted")
    plt.tight_layout()
    plt.savefig('actual_vs_predicted.png')
    plt.close()
else:
    feature_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()
    if target_col in feature_cols:
        feature_cols.remove(target_col)
    X_unsup = df_clean[feature_cols].fillna(0)
    best_model, inertias, sil_scores = run_kmeans(X_unsup)
    model_results = {'inertias': inertias, 'silhouette_scores': sil_scores}
    
# Clustering Visualizations
plt.figure(figsize=(6,4))
plt.plot(range(2,2 + len(inertias)), inertias, marker='o')
plt.title("Elbow Plot")
plt.xlabel("Number of Clusters (k)")
plt.ylabel("Inertia")
plt.tight_layout()
plt.savefig('kmeans_elbow.png')
plt.close()

# Final output
print("\n========== Pipeline Execution Summary ==========")
print(f"Before shape: {df.shape} | After shape: {df_clean.shape}")
print(f"Detected problem type: {problem_type}")
if isinstance(model_results, list) and len(model_results) > 0:
    print(f"Models trained: {[m['model'] for m in model_results]}")
    print(f"Best model pipeline: {best_model}")
else:
    print("Unsupervised Clustering completed successfully.")
print("\nFirst 2 rows of df_clean:")
print(df_clean.head(2))

Dataset shape:  (12000, 15)
Missing values summary:  {'video_id': 0, 'category': 0, 'duration_sec': 0, 'title_length': 0, 'thumbnail_face': 0, 'upload_hour': 0, 'upload_day': 0, 'tags_count': 0, 'description_length': 0, 'subscriber_count': 0, 'views': 0, 'likes': 0, 'comments': 0, 'shares': 0, 'viral': 0}
Duplicate rows:  0
Numeric columns count:  12
Categorical columns count:  3
       duration_sec  title_length  thumbnail_face   upload_hour    tags_count  \
count  12000.000000  12000.000000    12000.000000  12000.000000  12000.000000   
mean     631.619583     54.524000        0.400250     11.539417     14.586833   
std      285.825411     14.950788        0.489969      6.914424      7.741422   
min       30.000000     10.000000        0.000000      0.000000      0.000000   
25%      408.000000     45.000000        0.000000      5.000000      9.000000   
50%      614.000000     55.000000        0.000000     12.000000     14.000000   
75%      822.000000     65.000000        1.000000 

/tmp/ipykernel_16/2484690951.py:54: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if pd.api.types.is_bool_dtype(series) or pd.api.types.is_object_dtype(series) or pd.api.types.is_categorical_dtype(series):


Model KNeighborsClassifier failed: name 'best_score' is not defined
Model Evaluation Results:
    accuracy  precision    recall        f1                 model
0  0.978333   0.978497  0.978333  0.977974    LogisticRegression
1  0.865000   0.867181  0.865000  0.842289  KNeighborsClassifier
Model DecisionTreeClassifier failed: name 'best_score' is not defined
Model Evaluation Results:
    accuracy  precision    recall        f1                   model
0  1.000000   1.000000  1.000000  1.000000  DecisionTreeClassifier
1  0.978333   0.978497  0.978333  0.977974      LogisticRegression
2  0.865000   0.867181  0.865000  0.842289    KNeighborsClassifier
Model RandomForestClassifier failed: name 'best_score' is not defined
Model Evaluation Results:
    accuracy  precision    recall        f1                   model
0  1.000000   1.000000  1.000000  1.000000  RandomForestClassifier
1  1.000000   1.000000  1.000000  1.000000  DecisionTreeClassifier
2  0.978333   0.978497  0.978333  0.977974     